In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus

username = "root"
password = quote_plus("Rohini@&1912")  # important!
host = "localhost"
database = "company_dw"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")
# ===============================
# LOAD ALL TABLES
# ===============================

dim_date = pd.read_sql("SELECT * FROM dim_date", engine)
dim_customer = pd.read_sql("SELECT * FROM dim_customer", engine)
dim_product = pd.read_sql("SELECT * FROM dim_product", engine)
dim_employee = pd.read_sql("SELECT * FROM dim_employee", engine)
dim_channel = pd.read_sql("SELECT * FROM dim_channel", engine)
fact_sales = pd.read_sql("SELECT * FROM fact_sales", engine)

print("Loaded Successfully!")

Loaded Successfully!


In [4]:
# Remove duplicates
dim_date = dim_date.drop_duplicates()

# Fix date format
dim_date['full_date'] = pd.to_datetime(dim_date['full_date'])

# Remove invalid years
dim_date = dim_date[dim_date['year'] >= 2020]

print("dim_date cleaned:", dim_date.shape)

dim_date cleaned: (1000, 10)


In [5]:
dim_customer = dim_customer.drop_duplicates()

# Handle missing values
dim_customer['customer_name'] = dim_customer['customer_name'].fillna("Unknown")
dim_customer['segment'] = dim_customer['segment'].fillna("Consumer")

# Fix age (remove unrealistic values)
dim_customer = dim_customer[(dim_customer['age'] >= 18) & 
                            (dim_customer['age'] <= 80)]

# Standardize text
dim_customer['gender'] = dim_customer['gender'].str.title()

print("dim_customer cleaned:", dim_customer.shape)

dim_customer cleaned: (500, 9)


In [6]:
dim_product = dim_product.drop_duplicates()

# Remove negative prices
dim_product = dim_product[dim_product['price'] > 0]
dim_product = dim_product[dim_product['cost'] >= 0]

# Ensure price >= cost
dim_product = dim_product[dim_product['price'] >= dim_product['cost']]

print("dim_product cleaned:", dim_product.shape)

dim_product cleaned: (200, 6)


In [7]:
dim_employee = dim_employee.drop_duplicates()

dim_employee['hire_date'] = pd.to_datetime(dim_employee['hire_date'])

# Remove future hire dates
dim_employee = dim_employee[dim_employee['hire_date'] <= pd.Timestamp.today()]

print("dim_employee cleaned:", dim_employee.shape)

dim_employee cleaned: (100, 5)


In [8]:
dim_channel = dim_channel.drop_duplicates()

# Standardize text
dim_channel['channel_name'] = dim_channel['channel_name'].str.title()

print("dim_channel cleaned:", dim_channel.shape)

dim_channel cleaned: (3, 2)


In [9]:
fact_sales = fact_sales.drop_duplicates()

# Remove invalid quantities
fact_sales = fact_sales[fact_sales['quantity'] > 0]

# Remove negative values
fact_sales = fact_sales[fact_sales['sales_amount'] >= 0]
fact_sales = fact_sales[fact_sales['cost_amount'] >= 0]

# Recalculate profit safely
fact_sales['profit'] = fact_sales['sales_amount'] - fact_sales['cost_amount']

# Remove extreme outliers (IQR method)
Q1 = fact_sales['sales_amount'].quantile(0.25)
Q3 = fact_sales['sales_amount'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

fact_sales = fact_sales[(fact_sales['sales_amount'] >= lower) &
                        (fact_sales['sales_amount'] <= upper)]

print("fact_sales cleaned:", fact_sales.shape)

fact_sales cleaned: (3000, 11)


In [10]:
# Remove orphan records
fact_sales = fact_sales[
    fact_sales['customer_id'].isin(dim_customer['customer_id']) &
    fact_sales['product_id'].isin(dim_product['product_id']) &
    fact_sales['employee_id'].isin(dim_employee['employee_id']) &
    fact_sales['channel_id'].isin(dim_channel['channel_id']) &
    fact_sales['date_id'].isin(dim_date['date_id'])
]

print("Foreign key validation complete:", fact_sales.shape)

Foreign key validation complete: (3000, 11)


In [11]:
fact_sales['profit_margin'] = (fact_sales['profit'] / 
                               fact_sales['sales_amount']) * 100

fact_sales['avg_unit_price'] = fact_sales['sales_amount'] / fact_sales['quantity']

fact_sales['discounted_sales'] = fact_sales['sales_amount'] * (1 - fact_sales['discount'])

In [12]:
with pd.ExcelWriter("cleaned_company_dw.xlsx") as writer:
    dim_date.to_excel(writer, sheet_name="dim_date", index=False)
    dim_customer.to_excel(writer, sheet_name="dim_customer", index=False)
    dim_product.to_excel(writer, sheet_name="dim_product", index=False)
    dim_employee.to_excel(writer, sheet_name="dim_employee", index=False)
    dim_channel.to_excel(writer, sheet_name="dim_channel", index=False)
    fact_sales.to_excel(writer, sheet_name="fact_sales", index=False)

print("✅ All cleaned tables exported successfully!")

✅ All cleaned tables exported successfully!


In [3]:
dim_date.to_sql("dim_date_cleaned", engine, if_exists="replace", index=False)
dim_customer.to_sql("dim_customer_cleaned", engine, if_exists="replace", index=False)
dim_product.to_sql("dim_product_cleaned", engine, if_exists="replace", index=False)
dim_employee.to_sql("dim_employee_cleaned", engine, if_exists="replace", index=False)
dim_channel.to_sql("dim_channel_cleaned", engine, if_exists="replace", index=False)
fact_sales.to_sql("fact_sales_cleaned", engine, if_exists="replace", index=False)

print("✅ Cleaned tables saved to MySQL!")

✅ Cleaned tables saved to MySQL!
